# INFO 3350/6350

## Lecture 17: Intro to LLMs: Zero-shot and few-shot learning

# Zero-shot learning 

Our goal is to learn what **zero-shot** (and few-shot) **learning** is, how to do it, and how to evaluate different prompting strategies.

## What is zero-shot learning and how is it different from fine-tuning?

In lecture 15, we saw how to **fine-tune** an existing pretrained language model by changing its weights in response to a new task. In contrast, the **zero-shot** paradigm leaves model weights untouched. This makes it much faster than fine-tuning, though zero-shot accuracy is sometimes lower.

In the zero-shot paradigm, the main idea is to construct an input to the model and then compare which label is most likely, all without changing model parameters. And since there's no training (i.e. changing model parameters), there's no train/test split. All data is treated as evaluation data here.

## Example task: predicting the genre of a reviewed book

Let's consider the book review genre classification task.  In this book review task, an *example* consists of a review's text, and its *label* is the genre of the book. Here's a sample in the dataset:
```
This series is quite seriously a joke
in the realm of vampire novels. Even
though it's targeted for a young
audience, there's really no excuse for
this poorly done series...
```
Its corresponding label is `fantasy`.

With zero-shot learning, we can't just feed examples into an LM the way we could with fine-tuning. We need to add additional text to the beginning and/or end of an example, because the model's parameters are not being changed. The **prompt** (also called a **template**) is all the extra text that we add around an example in order to get a useful output from an LLM. We'll see several different prompts and prompting strategies below.

With the current generation of powerful generative language models, this paradigm will often be the first thing to try when evaluating a new task.

## Setup

### LLaMA.cpp and Ollama

[LLaMA.cpp](https://github.com/ggml-org/llama.cpp) is a popular tool for running LLMs locally. LLaMA is a family of open source LLMs developed by Meta, and the ".cpp" is a nod to the fact that the library is written in C++. Despite the name, you can use LLaMA.cpp to run most generative LLMs you can find on HuggingFace, provided that they are available in a standard format. 

[Ollama](https://ollama.com) is a consumer grade command line tool for running LLMs. It's similar to LLaMA.cpp, but friendlier. You may use either of these tools to run an LLM server, and I'm fine with whatever you choose to do. In order to get started with one (or both), I have written [a tutorial](https://github.com/jam963/local-inference-tutorial) that covers local LLM inference using Ollama and Llama.cpp. 

In either case, you will need to use one of these tools to download and serve a model that your machine can run at a reasonable speed. For Ollama, you can browse available models [here](https://ollama.com/search). Some recommendations for Ollama: 
- `gemma3:270m`: Very small, very bad, but fine for learning how to put a prompt together. Can be fun to see how bad a language model can be at basic tasks!
- `qwen3:0.6b-q8_0`: Small, quantized, not that great. Will (probably) run on your machine. Has "thinking". 
- `gemma3:1b-it-qat`: quantized, pretty small, and pretty bad at most things
- `llama3.1:8b-instruct-q4_K_M`: 8-billion scale, quantized. 5GB in size, but will give you decent outputs. Don't run this unless you have a decent amount of RAM and/or a dedicated graphics card.

This notebook assumes that you have either Ollama or LLaMA.cpp up and running with at least one model. If you aren't confident with the command line and digging through documentation, I would recommend using Ollama. [This information](https://docs.ollama.com/api/openai-compatibility) will be helpful. Here, we are using `gemma3:1b-it-qat`, so you will need to download and serve that model with Ollama if you want to run this notebook without modifying any code.

In addition, you will need to install the `openai` package into your conda environment. We are not going to be using OpenAI models (though you can do so using this package). We are only going to use the OpenAI API to communicate with the model we will be running on our machine. 

### Setup: import packages

In [1]:
from openai import OpenAI
import gdown
import json
import numpy as np
import pandas as pd
import random
from tqdm.notebook import tqdm
from IPython.display import display, Markdown

## Prompting a language model using the OpenAI API 

In [2]:
client = OpenAI(
    base_url="http://localhost:11434/v1", # Assuming Ollama w/ default config
                                          # This will be different if using LLaMA.cpp

    # Required but ignored
    api_key="ollama"
)

We'll use the `client` above to communicate with our model. We've "pointed" the API to the correct port used by Ollama to serve models using `base_url`; if we were using OpenAI's models, we'd need a real OpenAI API key and we'd leave the `base_url` as None. 

Let's write a test prompt:

In [3]:
test_prompt = "Tell me about yourself."

In [4]:
test_response = client.chat.completions.create(
    model="gemma3:1b-it-qat",           # Specify which downloaded model we want to use. 
                                        # If using LLaMA.cpp, this is ignored if using llama-server
                                        # in the way I demonstrate in my tutorial.
    messages=[
        {
            "role": "user",             # Each message has a role and content 
            "content": test_prompt
        }
    ], 
)

Here's what that `test_response` looks like: 

In [5]:
test_response

ChatCompletion(id='chatcmpl-123', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Okay, here I go! I'm Gemma, a large language model created by the Gemma team at Google DeepMind. \n\nI'm an open-weights model, which means my weights are publicly available. This allows researchers and developers to use, study, and modify me. \n\nHere's a little bit more about what I can do:\n\n* **I can generate text:** I can write different kinds of creative text formats, like poems, code, scripts, musical pieces, email, letters, etc.  Just give me a bit of a prompt!\n* **I can answer your questions:** I've been trained on a massive amount of text data, so I can provide helpful and informative answers to a wide range of questions.\n* **I can follow instructions:** I'm pretty good at understanding instructions and completing requests. \n* **I'm still learning:** Like everyone else, I'm constantly being updated and improved!\n\n**Basically, I'm here to

If everything's working correctly, we should be able to see a model response like so:

In [6]:
test_response.choices[0].message.content

"Okay, here I go! I'm Gemma, a large language model created by the Gemma team at Google DeepMind. \n\nI'm an open-weights model, which means my weights are publicly available. This allows researchers and developers to use, study, and modify me. \n\nHere's a little bit more about what I can do:\n\n* **I can generate text:** I can write different kinds of creative text formats, like poems, code, scripts, musical pieces, email, letters, etc.  Just give me a bit of a prompt!\n* **I can answer your questions:** I've been trained on a massive amount of text data, so I can provide helpful and informative answers to a wide range of questions.\n* **I can follow instructions:** I'm pretty good at understanding instructions and completing requests. \n* **I'm still learning:** Like everyone else, I'm constantly being updated and improved!\n\n**Basically, I'm here to help you out with whatever you need – just let me know!** \n\n---\n\nDo you want me to tell you something more specific about myself?

We can change generation parameters that will affect the model's output: 

In [9]:
test_response = client.chat.completions.create(
    model="gemma3:270m-it-qat",    
    messages=[
        {
            "role": "user",        
            "content": test_prompt
        }
    ], 
    temperature=3.0, 
)

Temperature controls how the model picks the next token as it generates its response. Higher temperatures mean, generally, more "random" responses. You can mess with this parameter, but it is often best to use the recommended defaults for the model you are using. If we set temperature very high, like above, we get weird, nonsensical outputs: 

In [10]:
test_response.choices[0].message.content

'Certainly. Please share what it interested readers as well for clarification!  😊\n'

There are other generation parameters that will affect output, but temperature's probably the one that you'll encounter most frequently.

### A toy example
Before we get to how to set up zero-shot learning, let's use a simplified example to see how it works.

Pretrained language models take as input a sentence (give or take; perhaps much more text) and produce the most likely next token. Given some question, the model will produce a response by sampling from the most likely next tokens given the tokens in the prompt. 

We can see a simple example when we ask the language model : `"What kind of animal is a cerulean warbler?"`. The token "bird" is the [factually correct answer](https://www.allaboutbirds.org/guide/Cerulean_Warbler/overview), so we'd expect that the model would respond with `"bird"`.

Let's try to identify some animals using our language model with a simple, zero-shot prompting approach.

In [11]:
animal_names = ["cerulean warbler", "bottlenose dolphin", "fruit fly"]
animal_labels = ["bird", "mammal", "insect"]
animal_types = ["bird", "mammal", "insect", "fish"]

prompt_template = "Given the name of an animal, " \
                  "select the TYPE that best describes that animal.\n" \
                  "To do this task, you will be provided with the name of an ANIMAL. " \
                  "Respond with only one of the VALID TYPES listed below.\n" \
                  "VALID TYPES: {types}.\n" \
                  "ANIMAL: {animal}" \

def animal_classifier(client, 
                      prompt_template, 
                      classes, 
                      X, 
                      model="gemma3:1b-it-qat"):
    classes = ", ".join(classes)
    preds = []
    for animal in X:
        prompt = prompt_template.format(animal=animal, types=classes)
        print(prompt)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt_template.format(animal=animal, types=classes)
                }
            ]
        )
        prediction = response.choices[0].message.content.strip().lower()
        print(prediction, "\n")
        preds.append(prediction)
    return preds

predicted_animal_types = animal_classifier(client, prompt_template, animal_types, animal_names)

print("Predicted: ", predicted_animal_types)
print("Actual: ", animal_labels)


Given the name of an animal, select the TYPE that best describes that animal.
To do this task, you will be provided with the name of an ANIMAL. Respond with only one of the VALID TYPES listed below.
VALID TYPES: bird, mammal, insect, fish.
ANIMAL: cerulean warbler
bird 

Given the name of an animal, select the TYPE that best describes that animal.
To do this task, you will be provided with the name of an ANIMAL. Respond with only one of the VALID TYPES listed below.
VALID TYPES: bird, mammal, insect, fish.
ANIMAL: bottlenose dolphin
mammal 

Given the name of an animal, select the TYPE that best describes that animal.
To do this task, you will be provided with the name of an ANIMAL. Respond with only one of the VALID TYPES listed below.
VALID TYPES: bird, mammal, insect, fish.
ANIMAL: fruit fly
insect 

Predicted:  ['bird', 'mammal', 'insect']
Actual:  ['bird', 'mammal', 'insect']


## How to set up a zero-shot evaluation

This toy example isn't very interesting. Let's consider the book review task that we introduced earlier.

In [12]:
# Download prepared book review data
texts_url = 'https://drive.google.com/uc?id=1qEZ3k9fZa_KITSQtFlhHq7zImUbBYCY2'
labels_url = 'https://drive.google.com/uc?id=1d-6abYwcwKfbYYdVH7mys7IeF4j9BMXP'
texts_filename = 'book_review_texts.json'
labels_filename = 'book_review_labels.json'
gdown.download(texts_url, texts_filename, quiet=True)
gdown.download(labels_url, labels_filename, quiet=True)
# And now load the data
with open(texts_filename, 'r') as f:
  all_texts = json.load(f)
with open(labels_filename, 'r') as f:
  all_labels = json.load(f)

We've seen what an **example** (also called a document) is before. In this book review task, an example consists of a review's text, and its **label** is the genre of the book. Here's a sample example in the dataset:
```
This series is quite seriously a joke
in the realm of vampire novels. Even
though it's targeted for a young
audience, there's really no excuse for
this poorly done series...
```
Its corresponding label is `fantasy`.

But we can't just feed examples into an LM the way we could with fine-tuning. With **zero-shot learning**, we need to add additional text to the beginning and/or end of an example because the model's parameters are not being changed. Look at the instructions we wrote above: we are clearly specifying the task, leaving as little ambiguity about what we want as possible.

The **prompt** is all the extra text that we add around an example in order to get a useful output from an LLM. We'll see several different prompts throughout this part of the tutorial.

### Construct a dataset and task: `history/biography` vs. `poetry`

**Constructing the dataset and task**

For this task, we'll limit ourselves just to reviews of books in two genres: `history/biography` and `poetry`. We'll try to tell them apart.

First, we'll select just 100 documents that are labeled with `history_biography` or `poetry` ...

In [13]:
# This function keeps only the first n examples that are
# labeled with either label_1 or label_2 (with 50% from each label)
def subsample_two_classes(all_texts, all_labels, label_1, label_2, n):
  # Convert to numpy array for easier indexing
  all_texts = np.array(all_texts)
  all_labels = np.array(all_labels)
  # Take the first n/2 examples from each class in order to have a balanced task
  idxs_label_1 = np.where(all_labels == label_1)[0].tolist()
  idxs_label_2 = np.where(all_labels == label_2)[0].tolist()
  n_each_class = int(n/2)
  idxs_label_1 = idxs_label_1[:n_each_class]
  idxs_label_2 = idxs_label_2[:n_each_class]
  subset_idxs = idxs_label_1 + idxs_label_2
  # Shuffle the order of examples
  random.shuffle(subset_idxs)
  # Actually select the indexes and return the
  subset_texts = list(all_texts[subset_idxs])
  subset_labels = list(all_labels[subset_idxs])
  return subset_texts, subset_labels

In [14]:
# Now take only book reviews for books in these two genres
task_texts, task_labels = subsample_two_classes(all_texts, all_labels,
                                                'history_biography', 'poetry',
                                                n=100)

The labels are currently called `history_biography` and `poetry`. The actual names of the labels matter in this zero-shot setting because we are evaluating the likelihood of each (unlike in the fine-tuning paradigm, where the names do not matter). Let's replace the unwieldy underscore with a slash so that `history_biography` becomes `history/biography`.

In [15]:
# Map the original label to a new name for that label
# We will evaluate with the new name
original_label_to_new_name = {
    'history_biography': 'history/biography',
    'poetry': 'poetry',
}
# This is the list of choices the model will evaluate
#possible_choices = ['history/biography', 'poetry']
possible_choices = list(original_label_to_new_name.values())

# This function renames the labels
def rename_labels(labels, label_dict):
  return [label_dict[l] for l in labels]

# Now call the function to rename history_biography to history/biography
task_labels = rename_labels(task_labels, original_label_to_new_name)

In [16]:
possible_choices

['history/biography', 'poetry']

In [17]:
len(task_labels)

100

### Inference and Evaluation 



Let's start evaluating the model's performance on the task of telling book review genres apart.

An important choice here is the **prompt**. A prompt is a kind of template that maps an example to a specific text input for the language model. We'll shortly see some examples.

There are several steps we need to implement for evaluation:

1. Choose a prompt
2. Choose an example
3. Create a prompted example that we will give as input to the model
4. Get a response from the model for each sample using our formatted prompt.
5. Evaluate performance across our whole dataset.

We want our prompt to clearly state the task and give our model the best possible chance of choosing the correct answer. In our earlier example, we specified the task, what the model would see, and how we wanted our output. Let's try two different prompts and see how the model performs with each of them. 

In [19]:
simple_prompt = "What genre is the following text? Choose one of the choices below.\n" \
                "Text: {item}\n" \
                "Choices: {classes}"

detailed_prompt = "Your task is to identify the GENRE of a TEXT given the available OPTIONS." \
                  "Your response should be limited " \
                  "to only one of the available OPTIONS.\n" \
                  "TEXT: {item}\n" \
                  "OPTIONS: {classes}\n" \
                  "Given the above, which GENRE in OPTIONS best describes the TEXT? " \
                  "Respond with only one of the OPTIONS."

# almost exactly the same as our last classifier
def llm_classifier(client, template, classes, X, model="gemma3:1b-it-qat"):
    classes_str = ", ".join(classes)

    preds = []
    valid = []
    for sample in tqdm(X):
        prompt = template.format(item=sample, classes=classes)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user", 
                    "content": prompt
                }
            ]
        )
        prediction = response.choices[0].message.content.strip().lower()
        is_valid = int(prediction in classes)      # Is the output one of the classes?
        preds.append(prediction)
        valid.append(is_valid)
    
    return preds, valid

In [20]:
simple_preds, simple_validity = llm_classifier(client, simple_prompt, possible_choices, task_texts)

detailed_preds, detailed_validity = llm_classifier(client, detailed_prompt, possible_choices, task_texts)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [21]:
simple_validity_score = sum(simple_validity) / len(simple_validity)
print("Percent of samples with valid output using simple prompt: ", simple_validity_score)

detailed_validity_score = sum(detailed_validity) / len(detailed_validity)
print("Percent of samples with valid output using detailed prompt: ", detailed_validity_score)

Percent of samples with valid output using simple prompt:  0.0
Percent of samples with valid output using detailed prompt:  0.81


In [22]:
simple_preds[0]

'the correct answer is **poetry**.\n\nthe text explicitly states "love the poems," which is a common expression used to describe or discuss poetry.'

In [23]:
detailed_preds[0]

'poetry'

Prompting can be as much about guiding the model to the right answer as forcing the model to format its response correctly! With the simple prompt, we have absolutely horrible accuracy only because we can't easily extract the predicted label. Let's see what the accuracy is like when using the detailed prompt: 

In [24]:
overall_accuracy = (np.array(detailed_preds) == np.array(task_labels)).sum() / len(detailed_preds)

valid_indices = np.argwhere(detailed_validity).squeeze()
valid_pct = len(valid_indices) / len(detailed_preds)
valid_accuracy = (np.array(detailed_preds)[valid_indices] == np.array(task_labels)[valid_indices]).sum() \
                 / valid_indices.shape[0]

print("Overall accuracy: ", overall_accuracy)
print("Percent valid responses: ", valid_pct)
print("Accuracy, valid outputs only: ", valid_accuracy)

Overall accuracy:  0.56
Percent valid responses:  0.81
Accuracy, valid outputs only:  0.691358024691358


# Few-shot learning

So far, we've been doing *zero-shot* classification, meaning that we are only giving the model a description of the task and a single sample. However, as we saw in our reading for this week, LLMs are capable of *in-context learning*, meaning that they can learn to perform a task given a sufficient number of examples. 

In few-shot learning, we prompt the model with both instructions *and examples of a task being performed*. **This is not the same as training a model.** Unfortunately, you sometimes hear people say that they've "trained their ChatGPT to do {x}" through prompting. This is not the correct way to think about what is happening. In training, we are modifying the model's weights. Few-shot learning leaves the model's weights entirely unchanged. The model itself hasn't "learned" anything as a result of your prompt. Recall that a language model's job is to predict $p(token|context)$ for all tokens in the model's vocabulary. Few-shot learning is a method of creating $context$ that maximizes the probability of the tokens we want for a given task. Providing examples of a task being performed correctly is an established way of doing this. The "shot" in "few-shot" and "zero-shot" learning refer to the number of examples we give the model. 

Let's try this with our text classification task. Below, we will provide up to 5 examples of text-label pairs to the model, along with our (slightly modified) instructions. We'll compare performance across 

In [25]:
# Let's just take the first 5 as possible examples to give to our model.
example_texts = task_texts[:5]
example_labels = task_labels[:5]

eval_texts = task_texts[5:]
eval_labels = np.array(task_labels[5:])

few_shot_prompt = "Your task is to correctly identify the GENRE of a " \
                  "TEXT from a set of available OPTIONS. " \
                  "Your response should be limited " \
                  "to only one of the available OPTIONS.\n" \
                  "***\n" \
                  "EXAMPLES:\n" \
                  "{examples}" \
                  "***\n" \
                  "TEXT: {item}\n" \
                  "OPTIONS: {classes}\n" \
                  "GENRE: "

few_shot_example_template = "TEXT: \"{item}\"\nOPTIONS: {classes}\nGENRE: {label}\n\n"

In [26]:
example_labels

['poetry', 'poetry', 'history/biography', 'poetry', 'poetry']

In [27]:
def few_shot_llm_classifier(client, 
                            prompt_template, 
                            classes, 
                            X, 
                            example_template,
                            example_texts, 
                            example_labels,
                            num_examples=1,
                            model="gemma3:1b-it-qat"):
    classes_str = ", ".join(classes)

    assert len(example_texts) == len(example_labels) & len(example_texts) >= num_examples

    examples = ""
    for i in range(num_examples): 
        examples += example_template.format(
            item=example_texts[i], classes=classes_str, label=example_labels[i]
            )

    preds = []
    valid = []
    for sample in tqdm(X):
        prompt = prompt_template.format(item=sample, classes=classes, examples=examples)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user", 
                    "content": prompt
                }
            ]
        )
        prediction = response.choices[0].message.content.strip().lower()
        is_valid = int(prediction in classes)      # Is the output one of the classes?
        preds.append(prediction)
        valid.append(is_valid)
    
    return preds, valid

In [28]:
fs_results = {
    "num_examples": [], 
    "validity": [], 
    "accuracy_all": [], 
    "accuracy_valid": []
    }
for num_examples in [1, 3, 5]:
    preds, validity = few_shot_llm_classifier(client, 
                                              prompt_template=few_shot_prompt, 
                                              classes=possible_choices, 
                                              X=eval_texts,
                                              example_template=few_shot_example_template,
                                              example_texts=example_texts, 
                                              example_labels=example_labels,
                                              num_examples=num_examples)
    preds = np.array(preds)
    validity = np.array(validity)
    valid_idx = np.argwhere(validity).squeeze()

    accuracy_all = (preds == eval_labels).sum() / preds.shape[0]
    accuracy_valid = (preds[valid_idx] == eval_labels[valid_idx]).sum() / valid_idx.shape[0]
    validity_pct = validity.sum() / validity.shape[0]

    fs_results["num_examples"].append(round(num_examples, 3))
    fs_results["validity"].append(round(validity_pct, 3))
    fs_results["accuracy_all"].append(round(accuracy_all, 3))
    fs_results["accuracy_valid"].append(round(accuracy_valid, 3))


  0%|          | 0/95 [00:00<?, ?it/s]

  0%|          | 0/95 [00:00<?, ?it/s]

  0%|          | 0/95 [00:00<?, ?it/s]

In [29]:
pd.DataFrame(fs_results)

,num_examples,validity,accuracy_all,accuracy_valid
0,1,0.989,0.568,0.574
1,3,0.979,0.695,0.710
2,5,0.958,0.537,0.560


With a small model like this, one of the biggest gains we see from few-shot prompting is adherence to the desired formatting. The percent of outputs that follow our desired specification improves dramatically. We can also see that our accuracy improves quite a bit when providing even one example! 

Something else to note here is that accuracy doesn't always improve by providing more examples. You should probably curate the examples you provide in a prompt or sample them in a principled way. As you can see, more examples results in longer inference time, so you don't just want to throw a ton of random examples into your prompt without evidence that it leads to better results!

## A fun, open-ended task

One of the first things I did with GPT-3 (four years ago now!) was build a haiku generator. Poetry generation is an interesting task, as you read about for this week. Let's have some fun generating haikus with a small language model. 

As a first step, let's simplify our lives a little by building a base class that handles the text generation, assuming an Ollama server running locally on our machine.

In [30]:
class OllamaChatModel:
    def __init__(self, model): 
        self.client =  OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )
        self.model = model

    def generate(self, prompt,
                 temperature=1.0,   
                 frequency_penalty=0, # Penalizes tokens that have already been generated, 
                                      #     which might make for more interesting poems.
                 pretty=True,         # Make our output pretty :)
                 **kwargs
    ):

        completion = self.client.chat.completions.create(
            model=self.model,
            temperature=temperature,
            frequency_penalty=frequency_penalty,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            **kwargs
        )
        content = completion.choices[0].message.content
        
        if pretty:
            return self.prettify(content) 
        return completion.choices[0].message.content

    @staticmethod
    def prettify(text):
        return Markdown(text.replace("\n", "  \n"))

Let's check the default performance of a model when asked to generate a haiku. We'll use a different (and much larger, 8B parameter) model for this, Qwen 3. If you want to run this notebook on your machine and don't have the ability/desire to run this model, just replace `model_name` below with the model of your choosing. 

In [31]:
model_name = "qwen3:8b"

In [32]:
model = OllamaChatModel(model_name)

model.generate("/no_think Write a haiku about Cornell University.") # We add /no_think to "turn off" Qwen3's reasoning.
                                                                    # Reasoning is just text that the model generates in order
                                                                    # generate a better response to the user. 
                                                                    # Reasoning takes time, so we'll turn it off here.
                                                                   

<think>  
  
</think>  
  
Bulbs glow on Ithaca's hill,    
Quiet paths through ancient woods—    
Knowledge grows in stillness.

By default, we get what is (probably close to) a passable haiku.

But maybe we can make some more "authentic"-sounding haikus through few-shot prompting. I've collected a small sample of haikus by [Matsuo Bashō](https://en.wikipedia.org/wiki/Matsuo_Bash%C5%8D) (1644-1694), a Japanese poet renowned for his haikus. (Many of his poems do not conform to the 5-7-5 pattern that we're used to in English-language haiku). 

To see if we can get more interesting results from this model, we'll randomly sample a few of his poems, along with a subject matter annotation that I've written, and provide these in a prompt. 

In [33]:
import json

with open("basho_corpus.json", "r") as f:
    basho_corpus = json.load(f)

basho_corpus = {k: v.replace("/", "\n") for k, v in basho_corpus.items()} # Reformat our poems
                                                                          # Keys are subjects, values are poems

basho_corpus["pond"]

'an old silent pond...\na frog jumps into the pond,\nsplash! Silence again.'

We can make our `OllamachatModel` a subclass of a `HaikuGenerator` class that will add prompt formatting and haiku sampling functionality.

In [34]:
from collections import namedtuple


HaikuExample = namedtuple("HaikuExample", ["subject", "haiku"]) # for clarity


class HaikuGenerator(OllamaChatModel): 
    """
    Generates English language haikus!
    """
    def __init__(self,  
                 prompt_template: str, 
                 haiku_data: dict,
                 model: str ="gemma3:1b-it-qat"):
        super().__init__(model)

        # Prompt template must have "examples" and "subject" fields
        self.prompt_template = prompt_template
        self.haikus = [HaikuExample(k, v) for k, v in haiku_data.items()] 

    def sample_random_haikus(self, n: int) -> dict:
        """
        Sample n random haikus from self.haikus
        """
        assert n < len(self.haikus)
        idx_to_sample = np.random.randint(
            len(self.haikus), 
            size=n
            )
        return [self.haikus[i] for i in idx_to_sample]
    
    def format_examples(self, examples: list[HaikuExample]) -> str:
        """
        Format examples for inclusion in a prompt.
        """
        formatted_examples = ""
        for ex in examples:
            formatted_examples += f"Subject: \"{ex.subject}\"\nHaiku: \"{ex.haiku}\"\n"
        return formatted_examples

    def format_prompt(self, subject: str, num_examples: int):
        """
        Format self.prompt_template with a subject and 
        num_examples examples.
        """
        examples = self.sample_random_haikus(num_examples)
        formatted_examples = self.format_examples(examples)

        return self.prompt_template.format(
            subject=subject, examples=formatted_examples
            )
    
    def generate(self, subject: str, num_examples: int, verbose: bool = False, **kwargs):
        """
        Generate a haiku using num_examples examples. 
        If verbose is True, print the prompt.
        """
        prompt = self.format_prompt(subject, num_examples)
        if verbose:
            print(prompt)
        return super().generate(prompt, **kwargs)


In [35]:
haiku_prompt_template = "/no_think You are a poet. You write beautiful haikus. " \
                        "You can write a haiku about any subject. You are also a master imitator of " \
                        "poetic style.\n" \
                        "You will be provided with several example subjects and haikus. Your task is to " \
                        "write a beautiful haiku about the last subject, imitating the style of the " \
                        "example haikus you have seen.\n\n" \
                        "Examples:\n{examples}\n" \
                        "Now, write a haiku about the following subject, being sure to imitate the style of the examples you have seen. " \
                        "Respond only with your haiku and nothing else.\n" \
                        "Subject: {subject}\nHaiku:"

haiku_generator = HaikuGenerator(haiku_prompt_template, basho_corpus, model=model_name)

Let's generate without our few-shot prompting as a kind of "baseline":

In [36]:
subject = "language models"
model.generate(f"/no_think Write a haiku about {subject}")


<think>  
  
</think>  
  
Silent thoughts take form,    
words flow through circuits, learning    
to speak without a mouth.

In [37]:
haiku_generator.generate(subject, 3, verbose=True, temperature=1.0)

/no_think You are a poet. You write beautiful haikus. You can write a haiku about any subject. You are also a master imitator of poetic style.
You will be provided with several example subjects and haikus. Your task is to write a beautiful haiku about the last subject, imitating the style of the example haikus you have seen.

Examples:
Subject: "night"
Haiku: "awake at night
the sound of a water jar
cracking in the cold"
Subject: "halves"
Haiku: "don't imitate me;
it's as boring
as the two halves of a melon"
Subject: "snow"
Haiku: "first snow
falling on
the half-finished bridge"

Now, write a haiku about the following subject, being sure to imitate the style of the examples you have seen. Respond only with your haiku and nothing else.
Subject: language models
Haiku:


<think>  
  
</think>  
  
silent code hums    
language models breathe in    
echoes of our speech

Let's try some other subjects and maybe mess with temperature a bit. Suggestions?

In [ ]:
haiku_generator.generate("", 3, verbose=True, temperature=2.0)

/no_think You are a poet. You write beautiful haikus. You can write a haiku about any subject. You are also a master imitator of poetic style.
You will be provided with several example subjects and haikus. Your task is to write a beautiful haiku about the last subject, imitating the style of the example haikus you have seen.

Examples:
Subject: "fleas"
Haiku: "fleas, lice,
a horse peeing
near my pillow"
Subject: "midfield"
Haiku: "midfield
attached to nothing
the skylark sings"
Subject: "spring"
Haiku: "spring:
a hill without a name
veiled in morning mist"

Now, write a haiku about the following subject, being sure to imitate the style of the examples you have seen. Respond only with your haiku and nothing else.
Subject: linguistics
Haiku:


<think>  
  
</think>  
  
linguistics —    
tongues twist in the dark    
echoes of old speech